# 📊 Integrador — Relatório de Marcas Inadimplentes

Notebook pronto para rodar no Google Colab.  
Une **LivePDV** + **Efí (Pagamentos + PIX)** e gera tabela consolidada.

## ⚠️ Segurança
- Suas credenciais ficam nos **Secrets** do Colab (chave no painel lateral 🔑), nunca no notebook.
- O certificado `.p12` é enviado pelo upload temporário (vai pra `/content/`, **não fica salvo**).
- Ao terminar, feche o notebook → os arquivos somem.

## Como usar
1. Clique no menu **Runtime → Run all** OU execute célula por célula
2. Configure os **Secrets** (instruções na célula 2)
3. Faça upload do `.p12` na célula 3
4. Veja o relatório na célula final

## 1️⃣ Instalar dependências e baixar o código

In [52]:
!pip install -q requests beautifulsoup4 lxml python-dotenv tabulate
!wget -q https://raw.githubusercontent.com/leonardochor-hash/integrador/main/livepdv_client.py
!wget -q https://raw.githubusercontent.com/leonardochor-hash/integrador/main/efi_client.py
!wget -q https://raw.githubusercontent.com/leonardochor-hash/integrador/main/relatorio_inadimplentes.py
!wget -q https://raw.githubusercontent.com/leonardochor-hash/integrador/main/classificador.py
!wget -q https://raw.githubusercontent.com/leonardochor-hash/integrador/main/excel_inadimplentes.py
!ls *.py

classificador.py  excel_inadimplentes.py  relatorio_inadimplentes.py
efi_client.py	  livepdv_client.py	  run_efi_safe.py


## 2️⃣ Configurar Secrets (credenciais)

**Clique no ícone 🔑 (Secrets) no painel esquerdo do Colab** e adicione os seguintes secrets:

| Nome | Valor |
|------|-------|
| `LIVEPDV_USERNAME` | seu usuário do LivePDV |
| `LIVEPDV_PASSWORD` | sua senha do LivePDV |
| `EFI_CLIENT_ID` | Client_Id da conta RS (da tela Detalhar) |
| `EFI_CLIENT_SECRET` | Client_Secret da conta RS |

**Importante**: marque o toggle de cada secret para **"Acesso a este notebook"** (azul/ligado).

Quando terminar, execute a célula abaixo.

In [30]:
import os
from google.colab import userdata

# Carrega secrets de forma segura — não imprime valores!
secrets_necessarios = ['LIVEPDV_USERNAME', 'LIVEPDV_PASSWORD', 'EFI_CLIENT_ID', 'EFI_CLIENT_SECRET']
for nome in secrets_necessarios:
    try:
        valor = userdata.get(nome)
        if valor:
            os.environ[nome] = valor
            print(f'✓ {nome}: carregado ({len(valor)} caracteres)')
        else:
            print(f'✗ {nome}: VAZIO — adicione no painel 🔑')
    except Exception as e:
        print(f'✗ {nome}: ERRO — verifique se ativou "Acesso a este notebook"')

# Configurações fixas
os.environ['LIVEPDV_BASE_URL'] = 'https://expositores.moombox.com.br'
os.environ['EFI_SANDBOX'] = 'false'
os.environ['EFI_DIAS_ATRASO_MIN'] = '1'
print('\n✓ Variáveis de ambiente configuradas')

✓ LIVEPDV_USERNAME: carregado (7 caracteres)
✓ LIVEPDV_PASSWORD: carregado (10 caracteres)
✓ EFI_CLIENT_ID: carregado (50 caracteres)
✓ EFI_CLIENT_SECRET: carregado (54 caracteres)

✓ Variáveis de ambiente configuradas


## 3️⃣ Upload do certificado PIX (.p12)

Execute a célula abaixo. Vai aparecer um botão **"Escolher arquivos"** — selecione o arquivo `.p12` que você baixou do painel Efí.

O arquivo fica em memória temporária do Colab; quando você fechar o notebook, ele desaparece.

In [24]:
from google.colab import files
import os

print('Selecione o arquivo .p12 baixado da Efí (conta RS):')
uploaded = files.upload()

if uploaded:
    nome_arquivo = list(uploaded.keys())[0]
    caminho = f'/content/{nome_arquivo}'
    os.environ['EFI_PIX_CERT_PATH'] = caminho
    tamanho = os.path.getsize(caminho)
    print(f'\n✓ Certificado carregado: {nome_arquivo} ({tamanho} bytes)')
else:
    print('✗ Nenhum arquivo selecionado')

Selecione o arquivo .p12 baixado da Efí (conta RS):


Saving producao-328142-integradorrs.p12 to producao-328142-integradorrs (2).p12

✓ Certificado carregado: producao-328142-integradorrs (2).p12 (2657 bytes)


## 4️⃣ Converter .p12 → .pem (necessário para Python)

A biblioteca `requests` precisa do certificado em formato `.pem`. A célula abaixo converte automaticamente.

In [25]:
import os
import subprocess

p12 = os.environ.get('EFI_PIX_CERT_PATH')
if not p12 or not os.path.exists(p12):
    print('✗ Faça o upload do .p12 primeiro (célula anterior)')
else:
    pem = p12.replace('.p12', '.pem')
    # Converte sem senha (Efí emite sem senha)
    resultado = subprocess.run(
        ['openssl', 'pkcs12', '-in', p12, '-out', pem, '-nodes', '-passin', 'pass:'],
        capture_output=True, text=True
    )
    if os.path.exists(pem) and os.path.getsize(pem) > 0:
        os.environ['EFI_PIX_CERT_PATH'] = pem
        print(f'✓ Convertido: {pem} ({os.path.getsize(pem)} bytes)')
    else:
        print('✗ Erro na conversão:')
        print(resultado.stderr)

✓ Convertido: /content/producao-328142-integradorrs (2).pem (3667 bytes)


## 5️⃣ Teste rápido: relatório com dados fictícios (mock)

Antes de bater nas APIs reais, vamos ver se o código está funcionando com dados de exemplo.

In [ ]:
!python relatorio_inadimplentes.py --mock

## 6️⃣ Relatório REAL — conta Efí RS

Agora vamos rodar com seus dados reais. Isso vai:
1. Logar no LivePDV (usando seu usuário/senha)
2. Listar boletos vencidos na Efí Pagamentos
3. Listar cobranças PIX (cobv) vencidas
4. Cruzar CNPJs e mostrar tabela consolidada

Se aparecer erro, verifique se todos os Secrets estão corretos e se o certificado foi convertido.

In [50]:
!python relatorio_inadimplentes.py --real --csv

[1/4] Conectando ao LivePDV... OK (1267 expositores)
[2/4] Listando cobrancas vencidas na Efi... OK (0 CNPJs inadimplentes)
[3/4] Cruzando CNPJs LivePDV x Efi... OK
[4/4] Montando relatorio... OK (0 marcas no relatorio)

    RELATORIO DE MARCAS INADIMPLENTES — MODO REAL  
  Gerado em: 21/05/2026 15:54

Nenhuma marca inadimplente encontrada (ou erro na coleta).


In [ ]:
from google.colab import files

In [31]:
import os, requests, json
cid = os.environ.get('EFI_CLIENT_ID', '')
sec = os.environ.get('EFI_CLIENT_SECRET', '')
cert = os.environ.get('EFI_PIX_CERT_PATH', '')
print('CID len:', len(cid), 'prefix:', cid[:8] if cid else 'EMPTY')
print('SEC len:', len(sec))
print('CERT path:', cert, 'exists:', os.path.exists(cert) if cert else False, 'size:', os.path.getsize(cert) if cert and os.path.exists(cert) else 0)
r = requests.post('https://pix.api.efipay.com.br/oauth/token', json={'grant_type': 'client_credentials'}, auth=(cid, sec), cert=cert, timeout=30)
print('STATUS:', r.status_code)
print('BODY:', r.text[:500])

CID len: 50 prefix: Client_I
SEC len: 54
CERT path: /content/producao-328142-integradorrs (2).pem exists: True size: 3667
STATUS: 200
BODY: {"access_token":"eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ0eXBlIjoiYWNjZXNzX3Rva2VuIiwiY2xpZW50SWQiOiJDbGllbnRfSWRfNWNmMDcyMjVjYTkzZmI0YTQ5ZDc5MmU4YTNlYzMzOWI4NjgyOWMzNSIsImFjY291bnQiOjMyODE0MiwiYWNjb3VudF9jb2RlIjoiYmI2OWMyZGJjZjgwYTQ2NDg5NWY2ODRiOWJlNzUyMmEiLCJzY29wZXMiOlsiY29iLnJlYWQiLCJjb2Iud3JpdGUiLCJjb2JyLnJlYWQiLCJjb2JyLndyaXRlIiwiY29idi5yZWFkIiwiY29idi53cml0ZSIsImduLmJhbGFuY2UucmVhZCIsImduLmluZnJhY3Rpb25zLnJlYWQiLCJnbi5pbmZyYWN0aW9ucy53cml0ZSIsImduLmtleXMuYnVja2V0LnJlYWQiLCJnbi5vcGIuYXV0b2


In [40]:
%%writefile /tmp/teste.py
import os, requests, json
BASE = 'https://cobrancas.api.efipay.com.br'
tok = requests.post(BASE+'/v1/authorize', json={'grant_type':'client_credentials'}, auth=(os.environ['EFI_CLIENT_ID'], os.environ['EFI_CLIENT_SECRET']), timeout=30).json()['access_token']
H = {'Authorization': 'Bearer ' + tok}
for ct in ['billet', 'carnet', 'charge', 'one_step']:
  rr = requests.get(BASE+'/v1/charges', params={'charge_type':ct,'begin_date':'2025-12-01','end_date':'2026-05-21','limit':3}, headers=H, timeout=30)
  print(ct, '->', rr.status_code, '|', rr.text[:300])

Writing /tmp/teste.py


In [44]:
import os, requests, json
PIX = 'https://pix.api.efipay.com.br'
cert = os.environ['EFI_PIX_CERT_PATH']
tok = requests.post(PIX+'/oauth/token', json={'grant_type':'client_credentials'}, auth=(os.environ['EFI_CLIENT_ID'], os.environ['EFI_CLIENT_SECRET']), cert=cert, timeout=30).json()['access_token']
H = {'Authorization': 'Bearer ' + tok}
r = requests.get(PIX+'/v2/cobv', params={'inicio':'2025-05-21T00:00:00Z','fim':'2026-05-21T23:59:59Z','paginacao.itensPorPagina':100}, headers=H, cert=cert, timeout=30)
j = r.json() if r.status_code==200 else r.text
print('STATUS:', r.status_code)
if isinstance(j, dict):
  print('total:', len(j.get('cobs', [])))
  print(json.dumps(j, indent=2, ensure_ascii=False)[:2000])

STATUS: 200
total: 0
{
  "parametros": {
    "inicio": "2025-05-21T00:00:00.000Z",
    "fim": "2026-05-21T23:59:59.000Z",
    "paginacao": {
      "paginaAtual": 0,
      "itensPorPagina": 100,
      "quantidadeDePaginas": 0,
      "quantidadeTotalDeItens": 0
    }
  },
  "cobs": []
}


In [55]:
from google.colab import files
up = files.upload()
arq = list(up.keys())[0]
print('Arquivo carregado:', arq)

Saving 1307bbb3b9ebd4ba21135400131026f61779384512277.xls to 1307bbb3b9ebd4ba21135400131026f61779384512277 (1).xls
Arquivo carregado: 1307bbb3b9ebd4ba21135400131026f61779384512277 (1).xls


In [61]:
%%bash
for f in excel_inadimplentes.py relatorio_inadimplentes.py classificador.py efi_client.py livepdv_client.py; do
  curl -sH 'Accept: application/vnd.github.raw' "https://api.github.com/repos/leonardochor-hash/integrador/contents/$f?ref=main" -o $f
    echo "$(stat -c '%s' $f) bytes $f"
    done
    head -3 classificador.py

7688 bytes excel_inadimplentes.py
13333 bytes relatorio_inadimplentes.py
2184 bytes classificador.py
18837 bytes efi_client.py
14275 bytes livepdv_client.py
"""
classificador.py


In [62]:
!python relatorio_inadimplentes.py --real --excel "$arq" --csv

[1/4] Conectando ao LivePDV... OK (1267 expositores)
[2/4] Lendo planilha Excel (1307bbb3b9ebd4ba21135400131026f61779384512277 (1).xls)... OK (10 CNPJs inadimplentes)
[3/4] Cruzando CNPJs LivePDV x Efi... OK
[4/4] Montando relatorio... OK (10 marcas no relatorio)

    RELATORIO DE MARCAS INADIMPLENTES — MODO REAL  
  Gerado em: 21/05/2026 18:13

╭─────┬─────────────────────┬────────────────────┬────────┬───────────────┬────────────┬──────────┬────────────────────────────────╮
│   # │ Marca               │ CNPJ               │   Dias │ Em aberto     │ Bloqueio   │ Classe   │ Acao sugerida                  │
├─────┼─────────────────────┼────────────────────┼────────┼───────────────┼────────────┼──────────┼────────────────────────────────┤
│   1 │ CNPJ 20439036000183 │ 20.439.036/0001-83 │     28 │ R$   2.180,00 │ nenhum     │ ?        │ bloqueio_recebimento (nivel 1) │
│   2 │ CNPJ 20680129000103 │ 20.680.129/0001-03 │     22 │ R$   1.390,00 │ nenhum     │ ?        │ bloqueio_recebimento

In [51]:
import os,requests,json
B='https://cobrancas.api.efipay.com.br'
t=requests.post(B+'/v1/authorize',json={'grant_type':'client_credentials'},auth=(os.environ['EFI_CLIENT_ID'],os.environ['EFI_CLIENT_SECRET']),timeout=30).json()['access_token']
H={'Authorization':'Bearer '+t}
anos=[('2020-01-01','2020-12-31'),('2021-01-01','2021-12-31'),('2022-01-01','2022-12-31'),('2023-01-01','2023-12-31'),('2024-01-01','2024-12-31'),('2025-01-01','2025-12-31'),('2026-01-01','2026-12-31')]
res=[(b,e,ct,requests.get(B+'/v1/charges',params={'charge_type':ct,'begin_date':b,'end_date':e,'limit':100},headers=H,timeout=30)) for (b,e) in anos for ct in ['billet','carnet']]
[print(b,e,ct,'->',r.status_code,'qtd=',(r.json().get('data',[]) if r.status_code==200 else r.text[:100])[:0] or len(r.json().get('data',[])) if r.status_code==200 else r.text[:80]) for (b,e,ct,r) in res]

2020-01-01 2020-12-31 billet -> 200 qtd= 0
2020-01-01 2020-12-31 carnet -> 200 qtd= 0
2021-01-01 2021-12-31 billet -> 200 qtd= 0
2021-01-01 2021-12-31 carnet -> 200 qtd= 0
2022-01-01 2022-12-31 billet -> 200 qtd= 0
2022-01-01 2022-12-31 carnet -> 200 qtd= 0
2023-01-01 2023-12-31 billet -> 200 qtd= 0
2023-01-01 2023-12-31 carnet -> 200 qtd= 0
2024-01-01 2024-12-31 billet -> 200 qtd= 0
2024-01-01 2024-12-31 carnet -> 200 qtd= 0
2025-01-01 2025-12-31 billet -> 200 qtd= 0
2025-01-01 2025-12-31 carnet -> 200 qtd= 0
2026-01-01 2026-12-31 billet -> 200 qtd= 0
2026-01-01 2026-12-31 carnet -> 200 qtd= 0


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [46]:
!python /tmp/teste_ids.py

  File "/tmp/teste_ids.py", line 7
    print(cid, '->', r.status_code, '|', r.text[:400])
IndentationError: unexpected indent


## 7️⃣ Baixar o CSV gerado (opcional)

Se você quiser abrir o resultado no Excel:

In [ ]:
from google.colab import files
import glob

csvs = glob.glob('relatorios/*.csv')
if csvs:
    mais_recente = max(csvs, key=lambda f: os.path.getmtime(f))
    print(f'Baixando: {mais_recente}')
    files.download(mais_recente)
else:
    print('Nenhum CSV gerado. Rode o relatório com --csv primeiro.')

## 🔒 Limpeza ao terminar

Quando acabar, feche a aba do Colab. Os arquivos temporários (.p12, .pem) são apagados automaticamente.

Os **Secrets** continuam salvos na sua conta Google — você pode revogá-los a qualquer momento no painel 🔑.

---

## 💡 Próximos passos

Para usar a segunda conta (BS), basta:
1. Trocar os valores de `EFI_CLIENT_ID` e `EFI_CLIENT_SECRET` no painel 🔑
2. Fazer upload do `.p12` da conta BS
3. Re-executar as células 5 em diante